In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [24]:
# Import cleaned data into a dataframe
df = pd.read_csv('../data/Squirrel_dataset_cleaned.csv')
df.head()

,X,Y,Unique Squirrel ID,Hectare,Shift,Date,Hectare Squirrel Number,Age,Primary Fur Color,Highlight Fur Color,...,Other Activities,Kuks,Quaas,Moans,Tail flags,Tail twitches,Approaches,Indifferent,Runs from,Other Interactions
0,-73.974281,40.775534,11B-PM-1014-08,11B,PM,2018-10-14,8,Adult,Gray,NaN,...,NaN,False,False,False,False,False,False,False,False,NaN
1,-73.970268,40.776213,13E-AM-1017-05,13E,AM,2018-10-17,5,Adult,Gray,Cinnamon,...,NaN,False,False,False,False,False,False,False,False,NaN
2,-73.954120,40.793181,36H-AM-1010-02,36H,AM,2018-10-10,2,Adult,Gray,NaN,...,NaN,False,False,False,False,False,False,False,False,NaN
3,-73.958269,40.791737,33F-AM-1008-02,33F,AM,2018-10-08,2,Adult,Gray,NaN,...,NaN,False,False,False,False,False,False,True,False,NaN
4,-73.967429,40.782972,21C-PM-1006-01,21C,PM,2018-10-06,1,Adult,Gray,NaN,...,NaN,False,False,False,True,True,False,False,False,NaN


In [25]:
df.columns

Index(['X', 'Y', 'Unique Squirrel ID', 'Hectare', 'Shift', 'Date',
       'Hectare Squirrel Number', 'Age', 'Primary Fur Color',
       'Highlight Fur Color', 'Combination of Primary and Highlight Color',
       'Color notes', 'Location', 'Above Ground Sighter Measurement',
       'Specific Location', 'Running', 'Chasing', 'Climbing', 'Eating',
       'Foraging', 'Other Activities', 'Kuks', 'Quaas', 'Moans', 'Tail flags',
       'Tail twitches', 'Approaches', 'Indifferent', 'Runs from',
       'Other Interactions'],
      dtype='object')

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2959 entries, 0 to 2958
Data columns (total 30 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   X                                           2959 non-null   float64
 1   Y                                           2959 non-null   float64
 2   Unique Squirrel ID                          2959 non-null   object 
 3   Hectare                                     2959 non-null   object 
 4   Shift                                       2959 non-null   object 
 5   Date                                        2959 non-null   object 
 6   Hectare Squirrel Number                     2959 non-null   int64  
 7   Age                                         2959 non-null   object 
 8   Primary Fur Color                           2959 non-null   object 
 9   Highlight Fur Color                         1905 non-null   object 
 10  Combination 

In [27]:
# Perform an inference task to try and predict the primary of the squirrel effectively.
# First some translations need to be done in order for a sklearn model to predict on the data
# 1. the Color column needs to be converted to numerical data

color_mapping_dict = {'Gray': 0, 'Cinnamon': 1, 'Black': 2}
df['Primary Fur Color'] = df['Primary Fur Color'].map(color_mapping_dict)
df['Primary Fur Color'].value_counts()

Primary Fur Color
0    2474
1     382
2     103
Name: count, dtype: int64

In [28]:
# 2. The dataset needs limited to only columns which will be important for the inference task at hand
columns_to_keep = ['X', 'Y', 'Shift', 'Age', 'Primary Fur Color', 'Location', 'Above Ground Sighter Measurement', 
                   'Running', 'Chasing', 'Climbing', 'Eating', 'Foraging', 'Kuks', 'Quaas', 'Moans', 'Tail flags',
                   'Tail twitches', 'Approaches', 'Indifferent', 'Runs from']
df = df[columns_to_keep]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2959 entries, 0 to 2958
Data columns (total 20 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   X                                 2959 non-null   float64
 1   Y                                 2959 non-null   float64
 2   Shift                             2959 non-null   object 
 3   Age                               2959 non-null   object 
 4   Primary Fur Color                 2959 non-null   int64  
 5   Location                          2959 non-null   object 
 6   Above Ground Sighter Measurement  2959 non-null   object 
 7   Running                           2959 non-null   bool   
 8   Chasing                           2959 non-null   bool   
 9   Climbing                          2959 non-null   bool   
 10  Eating                            2959 non-null   bool   
 11  Foraging                          2959 non-null   bool   
 12  Kuks  

In [29]:
# 3. Above Ground Sighter Measurement needs to translate all false values to 0's, and make the value an int
df.loc[df['Above Ground Sighter Measurement'] == 'FALSE', 'Above Ground Sighter Measurement'] = 0
df['Above Ground Sighter Measurement'].value_counts()

Above Ground Sighter Measurement
0      2116
10      166
20       84
15       71
2        55
3        52
5        51
30       44
4        42
25       33
6        32
1        30
8        30
40       25
50       19
7        19
12       16
13       11
35       10
28        7
18        5
100       5
45        4
9         4
17        3
60        3
14        2
24        2
23        2
65        2
11        2
43        2
16        2
33        1
31        1
80        1
0         1
180       1
55        1
70        1
19        1
Name: count, dtype: int64

In [30]:
# 4. Translate some columns with two values to a boolean column
df['Adult'] = df['Age'].map({'Adult': True, 'Juvenile': False})
df['PM Sighting'] = df['Shift'].map({'PM': True, 'AM': False})
df['Seen Above Ground'] = df['Location'].map({'Ground Plane': False, 'Above Ground': True})

df.drop(['Age', 'Shift', 'Location'], axis=1, inplace=True)

In [44]:
## Comparing specific models and metrics. 
# At this point, we have the data in a usable format to run the inference task of predicting Primary fur color
# First, let's define some methods to split test and train data, and evaluate models

def evaluate_model(model_df, model_type, test_size=0.25, verbose=True):
    df_copy = model_df.copy()
    
    if verbose: print('Splitting into test and train')
    X = df_copy.drop(columns=['Primary Fur Color'])
    y = df_copy['Primary Fur Color']
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=test_size)
    
    if verbose:
        print(f'Train Length = {len(X_train)}')
        print(f'Test Length = {len(X_test)}')
    
    if verbose: print('Training Model')
    model_type.fit(X_train, y_train)
    
    if verbose: print('Predicting Model\n')
    y_pred = model_type.predict(X_test)
    acc: float = float(accuracy_score(y_test, y_pred))
    if verbose: print(f'Accuracy: {acc * 100:.2f}%')
    
    precision: float = float(precision_score(y_test, y_pred, average='macro'))
    if verbose: print(f'Macro Precision: {precision * 100:.2f}%')
    
    recall: float = float(recall_score(y_test, y_pred, average='macro'))
    if verbose: print(f'Macro Recall: {recall * 100:.2f}%\n')
    
    if verbose: 
        print('Confusion matrix, predictions in columns, actuals in rows\n'
          'Ordering: Gray; Cinnamon; Black')
        print(confusion_matrix(y_test, y_pred, labels=[0, 1, 2]))
    return acc, precision, recall

In [32]:
print('Evaluating Random Forest Classification Model')
model = RandomForestClassifier()
evaluate_model(df, model)

Evaluating Random Forest Classification Model
Splitting into test and train
Train Length = 2219
Test Length = 740
Training Model
Predicting Model

Accuracy: 84.19%
Macro Precision: 39.86%
Macro Recall: 36.23%

Confusion matrix, predictions in columns, actuals in rows
Ordering: Gray; Cinnamon; Black
[[614  18   0]
 [ 68   9   1]
 [ 30   0   0]]


In [33]:
print('Evaluating K Neighbors Model')
model = KNeighborsClassifier()
evaluate_model(df, model)

Evaluating K Neighbors Model
Splitting into test and train
Train Length = 2219
Test Length = 740
Training Model
Predicting Model

Accuracy: 82.57%
Macro Precision: 36.52%
Macro Recall: 35.60%

Confusion matrix, predictions in columns, actuals in rows
Ordering: Gray; Cinnamon; Black
[[602  29   1]
 [ 69   9   0]
 [ 30   0   0]]


In [34]:
print('Evaluating Gaussian Naive Bayes Model')
model = GaussianNB()
evaluate_model(df, model)

Evaluating Gaussian Naive Bayes Model
Splitting into test and train
Train Length = 2219
Test Length = 740
Training Model
Predicting Model

Accuracy: 20.68%
Macro Precision: 36.97%
Macro Recall: 39.01%

Confusion matrix, predictions in columns, actuals in rows
Ordering: Gray; Cinnamon; Black
[[ 85 430 117]
 [  3  60  15]
 [  3  19   8]]


In [35]:
print('Evaluating Decision Tree Model')
model = DecisionTreeClassifier()
evaluate_model(df, model)

Evaluating Decision Tree Model
Splitting into test and train
Train Length = 2219
Test Length = 740
Training Model
Predicting Model

Accuracy: 77.03%
Macro Precision: 45.46%
Macro Recall: 47.28%

Confusion matrix, predictions in columns, actuals in rows
Ordering: Gray; Cinnamon; Black
[[535  77  20]
 [ 46  29   3]
 [ 23   1   6]]


### Analysis of Model Results
- Each of the above models does a poor job at executing the inference task
- While each achieves a high accuracy (>75% in most cases excepting Naive Bayes), the algorithms clearly place on over emphasis on the most common color, Gray. 
- In particular, the K Neighbors and Random Forest algorithms place nearly all of their predictions on the Gray squirrel, which means that while their accuracy is high since most squirrels are gray, it's still not a great algorithm at all. 
- In order to combat this, I am going to perform some sampling of the dataset, so that there is equal representation for each color in the train and test dataset.

In [36]:
# Sample 382 (number of Cinnamon squirrels) Gray squirrels, to even things out a bit in this dataset
gray_squirrels = df.loc[df['Primary Fur Color'] == 0]
non_gray_squirrels = df.loc[df['Primary Fur Color'] != 0]

In [37]:
gray_squirrels = gray_squirrels.sample(n=382, random_state=42)
new_df = pd.concat([gray_squirrels, non_gray_squirrels])
new_df['Primary Fur Color'].value_counts()

Primary Fur Color
0    382
1    382
2    103
Name: count, dtype: int64

In [38]:
# Run the evaluation on the model with best accuracy, Random Forest
model = RandomForestClassifier()
evaluate_model(new_df, model)

Splitting into test and train
Train Length = 650
Test Length = 217
Training Model
Predicting Model

Accuracy: 48.39%
Macro Precision: 48.90%
Macro Recall: 42.66%

Confusion matrix, predictions in columns, actuals in rows
Ordering: Gray; Cinnamon; Black
[[48 48  4]
 [27 51  2]
 [22  9  6]]


In [39]:
# Evaluate with model with best Precision and Recall, Decision Tree
model = DecisionTreeClassifier()
evaluate_model(new_df, model)

Splitting into test and train
Train Length = 650
Test Length = 217
Training Model
Predicting Model

Accuracy: 51.61%
Macro Precision: 51.47%
Macro Recall: 48.23%

Confusion matrix, predictions in columns, actuals in rows
Ordering: Gray; Cinnamon; Black
[[51 41  8]
 [27 49  4]
 [11 14 12]]


From this experiment, we can see that the overall accuracy decreases by a lot, but the precision and recall improves in both cases. This is a generally good sign for the model, but the models are still not effective at predicting what color squirrel is to be seen. 

This is likely due to the fact that there are not a lot of differences between each of the difference squirrel colors, which is to be expected. Likely the squirrels are similar species, or have similar tendencies due to their likely frequent interaction with one another. 

Finally, we will test to see if updating the ratio of train and testing data improves the quality of the model at all. We will run thison the Decision Tree classifier, as it performed best on the updated dataset.

In [50]:
train_test_splits = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
for split in train_test_splits:
    print('Test Size: {}'.format(split))
    model = DecisionTreeClassifier()
    acc, precision, recall = evaluate_model(df, model, test_size=split, verbose=False)
    print('Accuracy: {}%; Precision: {}%; Recall {}%'.format(round(acc * 100, 2), round(precision * 100, 2), round(recall * 100, 2)))
    print()

Test Size: 0.1
Accuracy: 76.69%; Precision: 47.72%; Recall 51.19%

Test Size: 0.15
Accuracy: 77.7%; Precision: 44.61%; Recall 45.01%

Test Size: 0.2
Accuracy: 76.86%; Precision: 42.7%; Recall 44.21%

Test Size: 0.25
Accuracy: 76.76%; Precision: 45.11%; Recall 45.74%

Test Size: 0.3
Accuracy: 74.77%; Precision: 40.19%; Recall 41.34%

Test Size: 0.35
Accuracy: 74.71%; Precision: 39.43%; Recall 39.7%

Test Size: 0.4
Accuracy: 75.51%; Precision: 42.61%; Recall 44.4%

Test Size: 0.45
Accuracy: 74.4%; Precision: 40.62%; Recall 41.64%

Test Size: 0.5
Accuracy: 72.77%; Precision: 39.85%; Recall 40.8%



The model seems to have the best accuracy, precision, and recall when using a split size of 10%. However, this might be due to overfitting and not because the model is actually doing better with that split size. Changing the train-test split size seems to not be impacting this model well. 

However, let's try this with the dataframe with the reduced Gray squirrel dataframe

In [51]:
for split in train_test_splits:
    print('Test Size: {}'.format(split))
    model = DecisionTreeClassifier()
    acc, precision, recall = evaluate_model(new_df, model, test_size=split, verbose=False)
    print('Accuracy: {}%; Precision: {}%; Recall {}%'.format(round(acc * 100, 2), round(precision * 100, 2), round(recall * 100, 2)))
    print()

Test Size: 0.1
Accuracy: 41.38%; Precision: 41.46%; Recall 40.79%

Test Size: 0.15
Accuracy: 48.09%; Precision: 45.48%; Recall 46.03%

Test Size: 0.2
Accuracy: 52.3%; Precision: 51.74%; Recall 52.04%

Test Size: 0.25
Accuracy: 51.61%; Precision: 50.97%; Recall 52.12%

Test Size: 0.3
Accuracy: 49.43%; Precision: 48.4%; Recall 46.19%

Test Size: 0.35
Accuracy: 52.96%; Precision: 52.77%; Recall 49.22%

Test Size: 0.4
Accuracy: 47.26%; Precision: 45.74%; Recall 44.18%

Test Size: 0.45
Accuracy: 50.38%; Precision: 47.59%; Recall 46.23%

Test Size: 0.5
Accuracy: 50.23%; Precision: 45.3%; Recall 46.28%



In this case, larger train-test splits seem to create marginally better results when using the dataframe with less Gray squirrel examples. Particular the split of 0.35 seems to be the best in this particular case.  